# Build Database (Testing-Stage)
Test building the schema, process the raw dataframes through cleaning functions and push them into a SQL database.

## Setup

In [1]:
import sys
import os
from sqlalchemy import create_engine

# Set root path
sys.path.append(os.path.abspath(".."))

from api_utils import run_batch_ingestion
from pipeline_utils import (
    clean_countries_db,
    clean_cities_db,
    clean_airports_db,
    clean_airlines_db,
    clean_aircraft_db,
    clean_flights,
    # enrich_dim_aircraft
)

## Initialising Database

In [2]:
# =========================================================================
# CHECK IF DATABASE EXISTS
# =========================================================================
# Define database name
DB = "airlines_warehouse.db"

# Define database path
DB_PATH = f"data/{DB}"

# Ensure the directory exists
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

# Check if the database file already exists
db_exists = os.path.exists(DB_PATH)

# Create SQLAlchemy engine connection
engine = create_engine(f"sqlite:///{DB_PATH}")

if not db_exists:
    print("Initial run detected: Creating database schema and loading base dimensions...")

    # Execute schema.sql file to build tables, PKs, and FKs
    with open("schema.sql", "r") as f:
        schema_sql = f.read()

    with engine.begin() as conn:
        # SQLite execution requires raw connection execution for multi-statement DDL scripts
        raw_conn = conn.connection
        cursor = raw_conn.cursor()
        cursor.executescript(schema_sql)
        raw_conn.commit()

    print("Schema created successfully!")

    # ================================
    # Ingest and load slow-changing dimension tables (First run only)
    print("Running batch ingestion from AirLabs API...")

    # Define endpoints and their specific requirements
    ingestion_plan = {
        'flights': {},
        'airportsDB': {},
        'airlinesDB': {},
        'citiesDB': {},
        'countriesDB': {},
        'fleetsDB': {'airline_icao': 'BAW'},
        #
        #
    }
    df = run_batch_ingestion(ingestion_plan, verbose=False)

    print("Transforming datasets...")
    # Create Independent Dimensions
    dim_countries = clean_countries_db(df['countriesDB'])
    dim_cities = clean_cities_db(df['citiesDB'])
    dim_airports = clean_airports_db(df['airportsDB'])
    dim_airlines = clean_airlines_db(df['airlinesDB'])
    dim_aircrafts = clean_aircraft_db(df['fleetsDB'])
    # dim_time ?
    # dim_schedule ?

    # Create Dependent Fact Table & Telemetry/Time Dimension
    fact_flights, dim_flight_position, df_live_aircraft_patch = clean_flights(df['flights'])

    # ================================
    # Load Dim & Fact Tables into SQL
    print("Loading dimension and fact tables into the database...")

    # Independent Dimensions first (so foreign keys are ready)
    dim_countries.to_sql('dim_country', engine, if_exists='append', index=False)
    dim_cities.to_sql('dim_city', engine, if_exists='append', index=False)
    dim_airports.to_sql('dim_airport', engine, if_exists='append', index=False)
    dim_airlines.to_sql('dim_airline', engine, if_exists='append', index=False)
    dim_aircrafts.to_sql('dim_aircraft', engine, if_exists='append', index=False)

    # Dependent Dimensions & Fact
    fact_flights.to_sql('fact_flight', engine, if_exists='append', index=False)
    dim_flight_position.to_sql('dim_flight_position', engine, if_exists='append', index=False)

    print("Initial setup complete! Database created and populated successfully.")

else:
    print("Existing database found. Skipping schema creation. Ready for incremental append.")

Initial run detected: Creating database schema and loading base dimensions...
Schema created successfully!
Running batch ingestion from AirLabs API...
Starting Batch Ingestion at 20260722_17...
Successfully ingested flights
Successfully ingested airportsDB
Successfully ingested airlinesDB
Successfully ingested citiesDB
Successfully ingested countriesDB
Successfully ingested fleetsDB
Transforming datasets...
Original records from /countries: 252
Unique countries for DIM_COUNTRIES: 252
Original records from /citiesDB: 10273
Records after dropping NaNs: 10273
Unique cities for DIM_CITIES: 10273
Original records from /airports: 23349
Unique airports for DIM_AIRPORT: 20878
Original records from /airlines: 6576
Count after keeping non-null 'icao_code' rows: 6259
Final unique airline records for DIM_AIRLINES: 6240
Original records from /fleets: 50
Dropped 0 because of missing airplane_hex.
Unique aircraft records for DIM_AIRCRAFT: 50
Original records from /flights: 8438
Processed 8438 raw mov

## First Look

In [3]:
import sqlite3
import pandas as pd

# Connect to your database
conn = sqlite3.connect("data/airlines_warehouse.db")

# Check what tables were created successfully
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in database:")
print(tables)

# Peek at the first few rows of flights fact table
print("\nPreview of fact_flight:")
fact_preview = pd.read_sql("SELECT * FROM fact_flight LIMIT 5;", conn)
print(fact_preview)

conn.close()

Tables in database:
                  name
0          dim_country
1             dim_city
2          dim_airport
3         dim_aircraft
4          dim_airline
5             dim_time
6  dim_flight_position
7          fact_flight

Preview of fact_flight:
   flight_id flight_number movement_type    status dep_delayed_min  \
0          1           138          None  en-route            None   
1          2          6677          None  en-route            None   
2          3           613          None  en-route            None   
3          4           803          None  en-route            None   
4          5          2404          None  en-route            None   

  arr_delayed_min       time_key origin_airport_id dest_airport_id  \
0            None  20260722_1524              EDDB            OJAI   
1            None  20260722_1524              SAAR            MDPC   
2            None  20260722_1524              KJFK            TNCA   
3            None  20260722_1524              K

In [ ]:
## Enrichment